In [2]:
import os
print(os.listdir("../data/processed"))

['st12000nm0008_q2_2026.csv.gz']


In [3]:
import pandas as pd

data = pd.read_csv("../data/processed/st12000nm0008_2026q1q2.csv.gz",
                   parse_dates=["date"])
print(data.shape, data["serial_number"].nunique(), int(data["failure"].sum()))
print(data["date"].min().date(), data["date"].max().date())

smart = [c for c in data.columns if c.startswith("smart_")]
print("\nworst missing-value rates:")
print(data[smart].isna().mean().sort_values(ascending=False).head(8))

print("\nduplicate drive-days:", data.duplicated(["serial_number", "date"]).sum())

key = ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw",
       "smart_198_raw", "smart_9_raw", "smart_194_raw"]
last = data.sort_values("date").groupby("serial_number").tail(1)
print("\nlast reading per drive, by failure (0 = healthy, 1 = failed):")
print(last.groupby("failure")[key].median())
print(last.groupby("failure")[key].mean())

(3335615, 28) 18652 274
2026-01-01 2026-06-30

worst missing-value rates:
smart_1_raw     0.000006
smart_3_raw     0.000006
smart_4_raw     0.000006
smart_5_raw     0.000006
smart_7_raw     0.000006
smart_9_raw     0.000006
smart_10_raw    0.000006
smart_12_raw    0.000006
dtype: float64

duplicate drive-days: 0

last reading per drive, by failure (0 = healthy, 1 = failed):
         smart_5_raw  smart_187_raw  smart_188_raw  smart_197_raw  \
failure                                                             
0                0.0            0.0   0.000000e+00            0.0   
1             1260.0          151.0   8.590983e+09           16.0   

         smart_198_raw  smart_9_raw  smart_194_raw  
failure                                             
0                  0.0      54586.0           33.0  
1                 16.0      53338.0           32.0  
         smart_5_raw  smart_187_raw  smart_188_raw  smart_197_raw  \
failure                                                          

In [5]:
import pandas as pd

data = pd.read_csv("../data/processed/st12000nm0008_2026q1q2.csv.gz",
                   parse_dates=["date"])

fail_dates = (data.loc[data["failure"] == 1, ["serial_number", "date"]]
              .rename(columns={"date": "fail_date"}))
print("failed drives:", fail_dates["serial_number"].nunique(), "rows:", len(fail_dates))

smart = [c for c in data.columns if c.startswith("smart_")]
bad = data[smart].isna().all(axis=1)
print("rows dropped:", int(bad.sum()),
      "| of which failure rows:", int(data.loc[bad, "failure"].sum()))

data = data[~bad].drop(columns=["smart_198_raw"]).reset_index(drop=True)

data.to_csv("../data/processed/st12000nm0008_2026q1q2_clean.csv.gz", index=False)
fail_dates.to_csv("../data/processed/fail_dates.csv", index=False)
print(data.shape)

failed drives: 274 rows: 274
rows dropped: 20 | of which failure rows: 6
(3335595, 27)
